# Landmark Detection — Click Nz, Auto-Detect Rest

Semi-manual landmark detection: the user clicks Nz (nasion) on the mesh,
then the system auto-detects Iz, LPA, RPA, and Cz from the mesh geometry.
This replaces unreliable sticker-based detection.

In [27]:
import numpy as np
import pyvista as pv

import cedalion
import cedalion.io
import cedalion.plots
from cedalion import units
from cedalion.geometry.photogrammetry.anonymization import (
    detect_landmarks_from_nasion,
    anonymize_scan,
    AnonymizationConfig,
    AnonymizationMethod,
)

pv.set_jupyter_backend("server")

## 1. Load Scan

In [28]:
SUBJECT_NUMBER = 12
SCANS_FOLDER = "/home/ma7/BA/PG_Subjects11-15"

scan_path = f"{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj"
surface = cedalion.io.read_einstar_obj(scan_path)

print(f"Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces")

Loaded: 624,121 vertices, 1,169,802 faces


## 2. Click Nz on the Mesh

Click the nasion (bridge of the nose, between the eyebrows) on the mesh.
The clicked point is stored in `nz_store`.

In [29]:
nz_store = {}

def on_pick(point):
    """Callback for surface point picking."""
    nz_store["point"] = np.array(point)
    print(f"Nz selected at: X={point[0]:.1f}, Y={point[1]:.1f}, Z={point[2]:.1f}")

# Use a pop-up window — callbacks don't work with the jupyter server backend
pvplt = pv.Plotter(notebook=False)
cedalion.plots.plot_surface(pvplt, surface, opacity=1.0)
pvplt.enable_surface_point_picking(
    callback=on_pick,
    left_clicking=True,
    show_point=True,
    point_size=20,
    color="red",
    show_message="Left-click Nz (nasion), then close the window",
)
pvplt.add_text("Left-click Nz (nasion), then close window", position="upper_left", font_size=14)
pvplt.show()

Nz selected at: X=147.0, Y=116.2, Z=483.8
Nz selected at: X=107.6, Y=151.1, Z=428.0


In [30]:
# If clicking didn't work, manually set Nz coordinates here.
# Hover over the nasion in the plot above to read coordinates, then uncomment:
# nz_store["point"] = np.array([X, Y, Z])

# Check if Nz was captured:
if "point" in nz_store:
    print(f"Nz stored: {nz_store['point']}")
else:
    print("Nz NOT stored yet. Either click above or set manually in this cell.")

Nz stored: [107.57456031 151.09103956 428.04108405]


## 3. Auto-Detect Remaining Landmarks

From the clicked Nz, the algorithm detects Cz (vertex), Iz (inion),
LPA (left preauricular), and RPA (right preauricular) using mesh geometry.

In [31]:
# Use stored Nz or fall back to manual coordinates for testing
if "point" in nz_store:
    nz_point = nz_store["point"]
else:
    # Fallback: approximate Nz for Subject11 (from visual inspection)
    # Replace with actual clicked coordinates
    nz_point = np.array([152.0, 120.0, 445.0])
    print(f"WARNING: Using fallback Nz. Click the mesh in the cell above first.")

print(f"Nz position: {nz_point}")

landmarks = detect_landmarks_from_nasion(surface, nz_point)

print(f"\nDetected landmarks:")
for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    print(f"  {label}: X={pos[0]:7.1f}, Y={pos[1]:7.1f}, Z={pos[2]:7.1f}")

Nz position: [107.57456031 151.09103956 428.04108405]

Detected landmarks:
  Nz: X=  107.6, Y=  151.1, Z=  428.0
  Iz: X=  118.4, Y=  -46.2, Z=  439.6
  Cz: X=  204.9, Y=   42.6, Z=  437.6
  LPA: X=  102.4, Y=   47.4, Z=  532.4
  RPA: X=   80.0, Y=   32.6, Z=  350.6


## 4. Visualize Landmarks

All 5 landmarks displayed as colored spheres on the mesh.

In [32]:
LANDMARK_COLORS = {
    "Nz": "red",
    "Iz": "blue",
    "Cz": "green",
    "LPA": "yellow",
    "RPA": "orange",
}

pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface, opacity=0.7)

for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    sphere = pv.Sphere(radius=5.0, center=pos)
    color = LANDMARK_COLORS.get(str(label), "white")
    pvplt.add_mesh(sphere, color=color)
    pvplt.add_point_labels(
        [pos + np.array([8, 0, 0])],
        [str(label)],
        font_size=16,
        text_color=color,
        bold=True,
        shape=None,
    )

pvplt.add_text("Detected Landmarks", position="upper_left", font_size=14)
pvplt.show()

Widget(value='<iframe src="http://localhost:45487/index.html?ui=P_0x7f2bbf64e290_9&reconnect=auto" class="pyvi…

## 5. Preview Facial Region Mask

Compute the T-shaped facial mask and visualize it as a red overlay.
Red vertices will be anonymized; grey vertices remain untouched.

In [33]:
from cedalion.geometry.photogrammetry.anonymization import (
    detect_facial_landmarks,
    get_facial_region_mask,
)
from cedalion.dataclasses import VTKSurface

# Detect facial landmarks (geometric estimates from anatomical landmarks)
facial_landmarks = detect_facial_landmarks(surface, landmarks)

# Compute the T-shaped facial mask
facial_mask = get_facial_region_mask(
    surface=surface,
    facial_landmarks=facial_landmarks,
    protected_points=landmarks,
    protection_radius=15.0 * units.mm,
)

print(f"Facial mask: {facial_mask.sum():,} / {len(facial_mask):,} vertices "
      f"({100 * facial_mask.sum() / len(facial_mask):.1f}%)")

# Visualize: red = will be anonymized, grey = untouched
vtk_surface = VTKSurface.from_trimeshsurface(surface)
pv_mesh = pv.wrap(vtk_surface.mesh)
colors = np.full((len(facial_mask), 3), 200, dtype=np.uint8)  # grey
colors[facial_mask] = [255, 50, 50]  # red
pv_mesh.point_data["mask_colors"] = colors

pvplt = pv.Plotter()
pvplt.add_mesh(pv_mesh, scalars="mask_colors", rgb=True, opacity=1.0)
pvplt.add_text("Facial Region (red = anonymized)", position="upper_left", font_size=14)
pvplt.show()

Facial mask: 58,519 / 624,121 vertices (9.4%)


Widget(value='<iframe src="http://localhost:45487/index.html?ui=P_0x7f2be2e23890_10&reconnect=auto" class="pyv…

## 6. Anonymize Facial Region

Apply noise-based anonymization to destroy facial detail while preserving
head shape and protected landmark/optode positions.

In [34]:
# --- Anonymization strength ---
# Higher = more aggressive. 80 iterations is a good default.
# Try 40 for mild, 120+ for very aggressive.
NOISE_ITERATIONS = 80

config = AnonymizationConfig(
    method=AnonymizationMethod.NOISE,
    noise_iterations=NOISE_ITERATIONS,
)
result = anonymize_scan(
    surface=surface,
    anatomical_landmarks=landmarks,
    config=config,
    interactive=False,
    validate=True,
)

disp = result.vertex_displacements
face_disp = disp[result.facial_mask]
print(f"Noise iterations: {NOISE_ITERATIONS}")
print(f"Facial vertices:  {result.facial_mask.sum():,}")
print(f"Max displacement:  {disp.max():.2f} mm")
print(f"Mean face disp:    {face_disp.mean():.2f} mm")
print(f"Median face disp:  {np.median(face_disp):.2f} mm")

Noise iterations: 80
Facial vertices:  58,519
Max displacement:  9.96 mm
Mean face disp:    1.08 mm
Median face disp:  0.96 mm


## 7. Original vs Anonymized

In [35]:
pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface, opacity=1.0)
pvplt.add_text("Original", position="upper_left", font_size=14)

pvplt.subplot(0, 1)
cedalion.plots.plot_surface(pvplt, result.anonymized_surface, opacity=1.0)
pvplt.add_text(f"NOISE ({NOISE_ITERATIONS} iter)", position="upper_left", font_size=14)

pvplt.link_views()
pvplt.show()

Widget(value='<iframe src="http://localhost:45487/index.html?ui=P_0x7f2bec707290_11&reconnect=auto" class="pyv…